In [0]:
bronze_table = (
    "real_time_catalogue."
    "bronze_realtime."
    "transactions"
)

silver_table = (
    "real_time_catalogue."
    "silver_realtime."
    "transactions"
)

silver_checkpoint = (
    "/Volumes/real_time_catalogue/"
    "bronze_realtime/"
    "landing_events/"
    "_checkpoints/silver_transactions"
)

bronze_stream = spark.readStream.table(bronze_table)

In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    to_timestamp
)

silver_stream = (
    bronze_stream

    # Conversion des types
    .withColumn(
        "transaction_timestamp",
        to_timestamp(col("transaction_date"))
    )
    .withColumn(
        "quantity",
        col("quantity").cast("integer")
    )
    .withColumn(
        "unit_price",
        col("unit_price").cast("double")
    )

    # Recalcul du montant
    .withColumn(
        "total_amount",
        (
            col("quantity") * col("unit_price")
        ).cast("double")
    )

    # Suppression des lignes invalides
    .filter(col("transaction_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("transaction_timestamp").isNotNull())
    .filter(col("quantity") > 0)
    .filter(col("unit_price") > 0)

    # Gestion des événements en retard et des doublons
    .withWatermark(
        "transaction_timestamp",
        "10 minutes"
    )
    .dropDuplicates(["transaction_id"])

    # Ajout de la date du nettoyage
    .withColumn(
        "processed_timestamp",
        current_timestamp()
    )

    # Sélection des colonnes finales
    .select(
        "transaction_id",
        "customer_id",
        "product_id",
        "quantity",
        "unit_price",
        "total_amount",
        "transaction_timestamp",
        "source_type",
        "ingestion_timestamp",
        "processed_timestamp",
        "source_file"
    )
)

In [0]:
silver_query = (
    silver_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        silver_checkpoint
    )
    .trigger(availableNow=True)
    .toTable(silver_table)
)

silver_query.awaitTermination()

print("Nettoyage Silver terminé.")

Nettoyage Silver terminé.


In [0]:
%sql
SELECT *
FROM real_time_catalogue.silver_realtime.transactions
ORDER BY processed_timestamp DESC;

transaction_id,customer_id,product_id,quantity,unit_price,total_amount,transaction_timestamp,source_type,ingestion_timestamp,processed_timestamp,source_file
RT-5C7FF6B821A5,C07618,P0924,5,325.49,1627.45,2026-09-07T12:55:09.931Z,realtime,2026-09-07T13:05:27.315Z,2026-09-07T13:59:02.734Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785709931_3.json
RT-FE607FE30AE2,C07160,P0308,4,46.7,186.8,2026-09-07T12:55:12.480Z,realtime,2026-09-07T13:05:27.315Z,2026-09-07T13:59:02.734Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785712480_4.json
RT-0975346145A1,C02638,P0462,2,141.61,283.22,2026-09-07T12:55:17.610Z,realtime,2026-09-07T13:05:27.315Z,2026-09-07T13:59:02.734Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785717610_6.json
RT-FC80185510BE,C03895,P0746,4,283.43,1133.72,2026-09-07T12:55:04.594Z,realtime,2026-09-07T13:05:27.315Z,2026-09-07T13:59:02.734Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785704594_1.json
RT-F3BE88BB2F0E,C09742,P0057,1,287.42,287.42,2026-09-07T12:55:20.095Z,realtime,2026-09-07T13:05:27.315Z,2026-09-07T13:59:02.734Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785720095_7.json
RT-E9CEA38E3006,C09621,P0108,4,127.45,509.8,2026-09-07T12:55:25.084Z,realtime,2026-09-07T13:05:27.315Z,2026-09-07T13:59:02.734Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785725084_9.json
RT-E9699EC1C14E,C02058,P0360,2,174.69,349.38,2026-09-07T12:55:27.542Z,realtime,2026-09-07T13:05:27.315Z,2026-09-07T13:59:02.734Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785727542_10.json
RT-BA87BA5D15AD,C05193,P0209,4,77.05,308.2,2026-09-07T12:55:14.976Z,realtime,2026-09-07T13:05:27.315Z,2026-09-07T13:59:02.734Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785714976_5.json
RT-89CC914D0E0E,C04791,P0454,3,362.72,1088.16,2026-09-07T12:55:07.349Z,realtime,2026-09-07T13:05:27.315Z,2026-09-07T13:59:02.734Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785707349_2.json
RT-6626AEBE3AF6,C06233,P0564,1,68.18,68.18,2026-09-07T12:55:22.592Z,realtime,2026-09-07T13:05:27.315Z,2026-09-07T13:59:02.734Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785722592_8.json


In [0]:
%sql
SELECT COUNT(*) AS silver_transactions
FROM real_time_catalogue.silver_realtime.transactions;

silver_transactions
10
